# JCP Fall 26 — Review: Weeks 2 and 3

This notebook goes with the review slides. Every section uses the same three cell types:

1. **Predict**: read the cell and write your guess in the `# Guess:` comment *before* you run it.
2. **Fix**: the cell is broken on purpose. Read the error, find the bug, fix it.
3. **Your Turn**: replace every `...` with your own code.

Examples use **`tips`** (restaurant bills). Your Turn cells use **`titanic`** (passengers on the Titanic). Same skills, different data, so you write the code yourself instead of copying it.

Keep each cell short. If you're stuck for more than 2 minutes, ask your partner, then ask us.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, confusion_matrix, accuracy_score, precision_score, recall_score

print("pandas version:", pd.__version__)

pandas version: 3.0.5


In [38]:
# We'll be using example datasets from Seaborn
tips = sns.load_dataset("tips")
titanic = sns.load_dataset("titanic")

---
## 0. Python Data Types and Dictionaries

**Predict:** what will each line print? Write your guesses first.

In [ ]:
# Guess: int, float, str, bool, list?
print(type(25))
print(type(3.5))
print(type("25"))
print(type(True))

<class 'int'>
<class 'float'>
<class 'str'>
<class 'bool'>


**Example:** a dictionary is labeled pairs. Run each cell and watch what comes back.

In [ ]:
menu = {
    "item": ["Latte", "Mocha", "Tea"],
    "price": [4.50, 5.25, 3.00]
}
menu["price"]

[4.5, 5.25, 3.0]

In [ ]:
menu_df = pd.DataFrame(menu)
menu_df

,item,price
0,Latte,4.50
1,Mocha,5.25
2,Tea,3.00


In [ ]:
# One column = Series. The whole table = DataFrame.
print(type(menu_df["price"]))
print(type(menu_df))

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>


In [ ]:
menu_df.dtypes

,0
item,object
price,float64


**Fix:** this cell has **two** bugs. Run it, fix the first error, run it again, fix the second.

In [ ]:
broken = {
    "item": ["Latte", "Mocha", "Tea"]
    "price": [4.50, 5.25]
}
pd.DataFrame(broken)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (350489066.py, line 2)

**Your Turn:** build a dictionary of 3 classes you're taking, with keys `"course"`, `"professor"`, and `"day"`. Turn it into a DataFrame and check its dtypes.

In [3]:
my_classes = {
    "course": ["CS 61A", "DATA C8", "MATH 54"],
    "professor": ["John DeNero", "Lisa Yan", "Ko Ohm"],
    "day": ["Monday", "Wednesday", "Friday"]
}
classes_df = pd.DataFrame(my_classes)
classes_df

,course,professor,day
0,CS 61A,John DeNero,Monday
1,DATA C8,Lisa Yan,Wednesday
2,MATH 54,Ko Ohm,Friday


In [4]:
classes_df.dtypes

course       str
professor    str
day          str
dtype: object

---
## 1. First Look at a Dataset

**Example:** four quick checks on `tips`.

In [5]:
tips.shape

(244, 7)

In [6]:
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [7]:
tips.dtypes

total_bill     float64
tip            float64
sex           category
smoker        category
day           category
time          category
size             int64
dtype: object

In [8]:
tips["day"].value_counts()

day
Sat     87
Sun     76
Thur    62
Fri     19
Name: count, dtype: int64

**Your Turn:** same four checks on `titanic`. How many rows? Which column tells you who survived? How many passengers were in each `class`?

In [9]:
titanic.shape

(891, 15)

In [10]:
titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [11]:
titanic['class'].value_counts()

class
Third     491
First     216
Second    184
Name: count, dtype: int64

In [15]:
titanic[(titanic['survived'] == 1)]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
875,1,3,female,15.0,0,0,7.2250,C,Third,child,False,NaN,Cherbourg,yes,True
879,1,1,female,56.0,0,1,83.1583,C,First,woman,False,C,Cherbourg,yes,False
880,1,2,female,25.0,0,1,26.0000,S,Second,woman,False,NaN,Southampton,yes,False
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True


---
## 2. Selecting and Filtering

**Predict:** for each cell, guess *Series or DataFrame*, or the exact value.

In [ ]:
# Guess:
tips["tip"].head(3)

In [ ]:
# Guess:
tips[["tip"]].head(3)

In [ ]:
# Guess:
tips.iloc[0, 1]

In [ ]:
# Guess:
tips.loc[0, "day"]

**Example:** filtering in two steps. First the condition, then the brackets.

In [ ]:
# Step 1: the condition is a column of True/False
(tips["size"] > 4).head()

In [ ]:
# Step 2: put it inside brackets to keep only the True rows
tips[tips["size"] > 4]

In [ ]:
# Two conditions: parentheses around EACH one
tips[(tips["size"] > 4) & (tips["day"] == "Sun")]

**Fix:** this should return Saturday tables with more than 4 people.

In [ ]:
tips[tips["size"] > 4 & tips["day"] == "Sat"]

**Your Turn:** find all female passengers in 1st class (`pclass == 1`). How many are there?

In [20]:
first_class_women = titanic[(titanic['pclass'] == 1) & (titanic['sex'] == 'female')]
len(first_class_women)

94

---
## 3. Mutability, Groupby, and Merge

**Predict:** after running the first cell, what will `tips.head(1)` show? The biggest tip ($10.00) or something else?

In [ ]:
tips.sort_values("tip", ascending=False)

In [ ]:
# Guess:
tips.head(1)

In [ ]:
# fix this!

**Fix:** this should rename `total_bill` to `bill`, then show the new column.

In [22]:
tips = tips.rename(columns={"total_bill": "bill"})
tips["bill"].head()

0    16.99
1    10.34
2    21.01
3    23.68
4    24.59
Name: bill, dtype: float64

**Example:** groupby. Split by `day`, take the mean of `tip` in each group, combine.

In [23]:
tips.groupby("day")["tip"].mean()

day
Thur    2.771452
Fri     2.734737
Sat     2.993103
Sun     3.255132
Name: tip, dtype: float64

In [24]:
# More than one summary at once
tips.groupby("day")["tip"].agg(["mean", "count"])

,mean,count
day,,
Thur,2.771452,62
Fri,2.734737,19
Sat,2.993103,87
Sun,3.255132,76


**Example:** merge. Watch the shape before and after.

In [25]:
day_info = pd.DataFrame({
    "day": ["Thur", "Fri", "Sat", "Sun"],
    "weekend": [False, False, True, True]
})
day_info

,day,weekend
0,Thur,False
1,Fri,False
2,Sat,True
3,Sun,True


In [26]:
tips_weekend = pd.merge(tips, day_info, on="day")
print("before:", tips.shape)
print("after: ", tips_weekend.shape)
tips_weekend.head(3)

before: (244, 7)
after:  (244, 8)


,bill,tip,sex,smoker,day,time,size,weekend
0,16.99,1.01,Female,No,Sun,Dinner,2,True
1,10.34,1.66,Male,No,Sun,Dinner,3,True
2,21.01,3.50,Male,No,Sun,Dinner,3,True


**Your Turn:** what fraction of passengers survived in each `pclass`? (Hint: `survived` is 0 or 1, so its mean is the survival rate.)

In [35]:
titanic.groupby('pclass')['survived'].mean()

pclass
1    0.629630
2    0.472826
3    0.242363
Name: survived, dtype: float64

---
## 4. Missing Values

`tips` has no missing values, so we'll make a copy and blank out 3 tips on purpose.

In [ ]:
tips_messy = tips.copy()
tips_messy.loc[[0, 5, 10], "tip"] = np.nan
tips_messy.isna().sum()

In [ ]:
# Option 1: drop the rows
print("before:", tips_messy.shape)
print("after: ", tips_messy.dropna().shape)

In [ ]:
# Option 2: impute with the median
median_tip = tips_messy["tip"].median()
tips_messy["tip"].fillna(median_tip).isna().sum()

**Fix:** this should remove the missing rows, so the count at the end should be 0.

In [ ]:
tips_messy.dropna()
tips_messy["tip"].isna().sum()

**Your Turn:** clean `titanic`.

1. Count missing values per column, then the *fraction* missing.
2. Make `titanic_clean`: drop the `deck` column, fill `age` with its median, then drop the rows still missing `embarked`.
3. Check that nothing is missing anymore.

In [45]:
titanic.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [ ]:
titanic.isna()

In [46]:
titanic_clean = titanic.drop(columns=['deck'])
titanic_clean["age"] = titanic_clean['age'].fillna(titanic_clean['age'].median())
titanic_clean = titanic_clean.dropna(subset=['embarked'])
titanic_clean.shape

(889, 14)

In [47]:
titanic_clean.isna().sum().sum()

np.int64(0)

---
## 5. Visualizing

**Example:** one numeric column → histogram. Two numeric columns → scatter plot.

In [ ]:
sns.histplot(tips, x="total_bill", bins=20)
plt.title("Distribution of Total Bill")
plt.show()

**Tinker:** change `bins=20` to `bins=4`, then `bins=80`. Which one hides the shape? Which one is just noise?

In [ ]:
sns.scatterplot(tips, x="total_bill", y="tip", hue="time", alpha=0.6)
plt.title("Bigger Bills, Bigger Tips?")
plt.show()

**Your Turn:** plot a histogram of `fare` from `titanic_clean`. It's heavily right-skewed, so add a `log_fare` column (use `np.log(... + 1)`, since some fares are 0) and plot that too. What changed?

In [ ]:
sns.histplot(titanic_clean, x=..., bins=30)
plt.title(...)
plt.show()

In [ ]:
titanic_clean["log_fare"] = ...
sns.histplot(titanic_clean, x="log_fare", bins=30)
plt.title("Distribution of log(Fare + 1)")
plt.show()

---
## 6. Regression: Predicting a Number

**Example:** predict `tip` from the bill, party size, and day. Run one cell at a time.

**Fix:** models only accept numbers. Run this, read the error, then move on to the next cell to see the fix.

In [39]:
LinearRegression().fit(tips[["total_bill", "day"]], tips["tip"])

ValueError: could not convert string to float: 'Sun'

In [40]:
# One-hot encode day: Thur is dropped and becomes the reference
tips_model = pd.get_dummies(tips, columns=["day"], drop_first=True, dtype=int)
tips_model[["total_bill", "size", "day_Fri", "day_Sat", "day_Sun"]].head()

,total_bill,size,day_Fri,day_Sat,day_Sun
0,16.99,2,0,0,1
1,10.34,3,0,0,1
2,21.01,3,0,0,1
3,23.68,2,0,0,1
4,24.59,4,0,0,1


In [41]:
X = tips_model[["total_bill", "size", "day_Fri", "day_Sat", "day_Sun"]]
y = tips_model["tip"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("train:", X_train.shape, " test:", X_test.shape)

train: (195, 5)  test: (49, 5)


In [42]:
reg = LinearRegression()
reg.fit(X_train, y_train)
pd.Series(reg.coef_, index=X.columns).round(3)

total_bill    0.092
size          0.254
day_Fri       0.035
day_Sat      -0.130
day_Sun       0.044
dtype: float64

**Predict:** the model's test MSE vs. a constant model that always guesses the average tip. Which is lower?

In [43]:
model_mse = mean_squared_error(y_test, reg.predict(X_test))
baseline_mse = mean_squared_error(y_test, [y_train.mean()] * len(y_test))
print("model MSE:   ", round(model_mse, 3))
print("baseline MSE:", round(baseline_mse, 3))

model MSE:    0.666
baseline MSE: 1.449


**Your Turn:** predict a passenger's `fare` (a number) from `titanic_clean`.

1. One-hot encode `embarked` (drop first, `dtype=int`). Check the new column names.
2. Use features `age`, `pclass`, `sibsp`, and the `embarked` columns.
3. Split 80/20 with `random_state=42`, fit a `LinearRegression`, and look at the coefficients.
4. Compare your test MSE to the baseline that always guesses the average fare.

In [50]:
titanic_reg = pd.get_dummies(titanic_clean, columns=['embarked'], drop_first=True, dtype=int)
titanic_reg[["age", "pclass", "sibsp", "embarked_Q"]].head()

,age,pclass,sibsp,embarked_Q
0,22.0,3,1,0
1,38.0,1,1,0
2,26.0,3,0,0
3,35.0,1,1,0
4,35.0,3,0,0


In [ ]:
X_f = titanic_reg[[...]]
y_f = titanic_reg[...]
X_train, X_test, y_train, y_test = train_test_split(..., test_size=..., random_state=...)
print("train:", X_train.shape, " test:", X_test.shape)

In [ ]:
fare_reg = ...
fare_reg.fit(..., ...)
pd.Series(fare_reg.coef_, index=X_f.columns).round(3)

In [ ]:
fare_mse = mean_squared_error(y_test, fare_reg.predict(...))
fare_baseline = mean_squared_error(y_test, [y_train.mean()] * len(y_test))
print("model MSE:   ", round(fare_mse, 3))
print("baseline MSE:", round(fare_baseline, 3))

*What does the `pclass` coefficient mean, in your words?* (Hint: `pclass` goes up by 1 when a passenger moves from 1st toward 3rd class.) ...

---
## Done: Push Your Work

Run these in your terminal (not in this notebook):

```
git add Week_Review.ipynb
git commit -m "review notebook"
git push origin YOUR-BRANCH-NAME
```